# CNN_VIT_BILSTM_CROSS_ATTENTION_BASED_TRAFFIC_MANAGEMENT_SYSTEM
## S11 — fine-tune YOLOv8s on IndiaTrafficNet's IDD bootstrap

Step **S11** of the build log. It fine-tunes the edge detector that feeds
everything downstream: counts to congestion labels (ADR-002), congestion labels
to MFSTNet, MFSTNet predictions into the PPO state vector.

**The dataset attached here was built by a committed script**, not by hand —
`scripts/prepare_idd.py` in the repository, from IDD Detection, applying
`indiatrafficnet/class_mapping.yaml`. Every decision it encodes is testable
there rather than remembered here:

* `rider` is **dropped** — a motorcyclist is a `rider` *on* a `motorcycle`, and
  counting both inflates every vehicle count by ~19% of all boxes, which would
  bias every congestion label the pipeline derives.
* `vehicle fallback`, `bicycle`, `traffic sign`, `traffic light` are **dropped,
  not merged** — merging teaches a category that does not exist in our label
  space.
* Images are downscaled to a 960 px long edge. YOLO trains at 640, so 1920 px
  costs disk and I/O for detail the network never sees. Labels are unaffected:
  YOLO coordinates are normalised.

**`e_rickshaw` has zero examples and that is expected.** No public dataset
assessed carries the class — not IDD, not the DataCluster sample. It is pending
item **P12**, decided by the first self-recorded clip, not here.

**Seed 42 throughout** (NFR-07). Results land in a CSV that gets committed to the
repository; nothing is transcribed from a screenshot.

In [ ]:
import os, sys, json, random, subprocess, shutil
from pathlib import Path

SEED = 42                      # NFR-07 — PyTorch, NumPy, Python
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

import numpy as np, torch
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print("python  ", sys.version.split()[0])
print("torch   ", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu     ", torch.cuda.get_device_name(0))
    print("memory  ", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")
else:
    raise SystemExit("NO GPU. Settings -> Accelerator -> GPU T4 x2.")

# HARD GATE. Kaggle's P100 is compute capability sm_60, and current PyTorch
# builds start at sm_70 — the card is UNUSABLE, not merely slow, and the only
# hint is a UserWarning that scrolls past while the run keeps going. Version 1
# of this notebook hit exactly that.
#
# Fail here, in fifteen seconds, rather than after an hour of quota.
major, minor = torch.cuda.get_device_capability(0)
supported = torch.cuda.get_arch_list()
print("capability", f"sm_{major}{minor}", "| build supports", supported)
if f"sm_{major}{minor}" not in supported:
    raise SystemExit(
        f"INCOMPATIBLE GPU: this card is sm_{major}{minor} and the installed "
        f"PyTorch supports {supported}. "
        f"Fix: Settings -> Accelerator -> GPU T4 x2 (sm_75). "
        f"The P100 cannot run this build at all."
    )

## The data

Ultralytics resolves labels by substituting `images` -> `labels` in the image
path, so the directory layout has to survive the move to Kaggle. This rewrites
`data.yaml` to the Kaggle path rather than assuming the one baked in at build
time — a stale absolute path is the usual reason a Kaggle run trains on nothing.

In [ ]:
INPUT = Path("/kaggle/input")

# DISCOVER the dataset rather than assert a guessed path. Version 1 and version 4
# both died on a hardcoded `/kaggle/input/<slug>` that did not exist, and the
# assertion said only that it was missing — not what WAS there, which is the one
# fact needed to fix it.
print("contents of /kaggle/input:")
candidates = sorted(INPUT.iterdir()) if INPUT.exists() else []
for entry in candidates:
    print("  ", entry.name)
if not candidates:
    raise SystemExit(
        "NOTHING is mounted under /kaggle/input. "
        "The dataset is attached per-notebook: open the editor, use the right-hand "
        "Input panel -> Add Input, and search for "
        "'indiatrafficnet-bootstrap-idd-yolo'. `kaggle kernels push` carries "
        "dataset_sources, but a Save & Run All from the UI uses whatever the "
        "draft has attached."
    )

# The dataset is whichever mounted directory actually holds the YOLO layout.
def find_yolo_root(base, max_depth=4):
    """Walk down for the directory holding the YOLO layout.

    Kaggle mounts a dataset at its mountSlug, which for this one is
    `datasets/idredk/indiatrafficnet-bootstrap-idd-yolo` — so /kaggle/input holds
    a single `datasets` directory and the data is THREE levels down. Two runs
    died asserting a flat path. Searching is robust to whatever slug Kaggle uses
    and costs nothing.
    """
    stack = [(base, 0)]
    while stack:
        directory, depth = stack.pop()
        if (directory / "data.yaml").exists() or (directory / "images").is_dir():
            return directory
        if depth < max_depth:
            try:
                stack.extend((child, depth + 1) for child in directory.iterdir() if child.is_dir())
            except PermissionError:
                pass
    return None

DATA_ROOT = find_yolo_root(INPUT)
if DATA_ROOT is None:
    raise SystemExit(
        f"searched {[d.name for d in candidates]} four levels deep and found no "
        f"data.yaml or images/. Attach 'indiatrafficnet-bootstrap-idd-yolo'."
    )
print("using", DATA_ROOT)

# Rebuild data.yaml against the Kaggle path; the committed one is a Windows path.
import yaml
names = ["car","motorcycle","auto_rickshaw","e_rickshaw","bus","truck","pedestrian","cattle"]
WORK = Path("/kaggle/working/idd")
WORK.mkdir(parents=True, exist_ok=True)
(WORK / "data.yaml").write_text(yaml.safe_dump({
    "path": str(DATA_ROOT),
    "train": "images/train", "val": "images/val", "test": "images/test",
    "nc": len(names), "names": names,
}, sort_keys=False))
print((WORK / "data.yaml").read_text())

for split in ("train","val","test"):
    n_img = len(list((DATA_ROOT/"images"/split).glob("*.jpg")))
    n_lab = len(list((DATA_ROOT/"labels"/split).glob("*.txt")))
    flag = "OK" if n_img == n_lab and n_img else "MISMATCH"
    print(f"  {split:<6} {n_img:>6} images {n_lab:>6} labels  {flag}")

## Class balance — read this before believing any mAP

`cattle` is the rarest class by a wide margin. FR-D08 requires per-class support
beside every metric for exactly this reason: an mAP computed from a few hundred
boxes swings wildly and means very little. The numbers below decide how the
final table must be read.

In [ ]:
from collections import Counter
support = {s: Counter() for s in ("train","val","test")}
for split in support:
    for label_file in (DATA_ROOT/"labels"/split).glob("*.txt"):
        for line in label_file.read_text().splitlines():
            if line.strip():
                support[split][names[int(line.split()[0])]] += 1

print(f"{'class':<16}" + "".join(f"{s:>10}" for s in support))
for name in names:
    print(f"{name:<16}" + "".join(f"{support[s][name]:>10,}" for s in support))
print("\ne_rickshaw at 0 is EXPECTED — see P12. Any other zero is a defect.")

## Train

`yolov8s` per PRD §12.3 — not `n` (too weak for small three-wheelers at
distance) and not `m` (does not meet the ≥10 fps edge budget in FR-D06).

`patience=20` stops early rather than burning quota on a plateau. Kaggle's free
tier is ~30 GPU-hours a week (ADR-013 rev 2), and that budget is shared with the
MFSTNet ablation.

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
results = model.train(
    data=str(WORK / "data.yaml"),
    epochs=60,
    imgsz=640,
    batch=16,
    seed=SEED,
    patience=20,
    project="/kaggle/working/runs",
    name="s11_yolov8s_idd",
    exist_ok=True,
    plots=True,
)

## Evaluate — per class, with support attached

A single mAP number is not reportable here (FR-D08). The table below is what
gets committed; the paper's table is generated from that CSV, never typed.

In [ ]:
import csv
metrics = model.val(data=str(WORK / "data.yaml"), split="test", plots=True)

# `class_result(i)` indexes into the classes that HAVE INSTANCES, not into the
# names list. `ap_class_index` maps each row back to its real class id.
#
# Version 8 of this notebook got that wrong. e_rickshaw has zero boxes, so it was
# absent from the results arrays, every class after it shifted up by one, and the
# published table reported mAP50 0.7288 for a class with 0 boxes and `nan` for
# cattle which had 183. Both impossible, both printed without complaint.
present = list(metrics.box.ap_class_index)
rows = []
for position, class_id in enumerate(present):
    p, r, ap50, ap = metrics.box.class_result(position)
    rows.append({
        "class": names[int(class_id)],
        "precision": round(float(p), 4), "recall": round(float(r), 4),
        "mAP50": round(float(ap50), 4), "mAP50_95": round(float(ap), 4),
        "test_boxes": support["test"][names[int(class_id)]], "evaluated": True,
    })
for index, name in enumerate(names):
    if index not in present:
        rows.append({"class": name, "precision": None, "recall": None,
                     "mAP50": None, "mAP50_95": None,
                     "test_boxes": support["test"][name], "evaluated": False})
rows.sort(key=lambda row: names.index(row["class"]))

out = Path("/kaggle/working/s11_detector_metrics.csv")
with out.open("w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

print(f"{'class':<16}{'P':>8}{'R':>8}{'mAP50':>9}{'mAP50-95':>10}{'boxes':>8}")
for row in rows:
    if row["evaluated"]:
        print(f"{row['class']:<16}{row['precision']:>8.3f}{row['recall']:>8.3f}"
              f"{row['mAP50']:>9.3f}{row['mAP50_95']:>10.3f}{row['test_boxes']:>8,}")
    else:
        print(f"{row['class']:<16}{'-':>8}{'-':>8}{'-':>9}{'-':>10}"
              f"{row['test_boxes']:>8,}   NOT EVALUATED (no instances)")
print(f"overall mAP50 {float(metrics.box.map50):.4f}  mAP50-95 {float(metrics.box.map):.4f}")
print(f"wrote {out} - commit this, do not retype it")

## Speed — FR-D06 requires ≥10 fps on the edge device

Measured on a Kaggle T4, which is **not** the deployment target. ADR-003 made
the edge node a laptop rather than a Jetson, and every latency figure must state
its measurement host. This number is a **proxy** and is labelled as one; it
establishes an upper bound, not compliance.

In [ ]:
import time
model.predict(source=str(DATA_ROOT/"images"/"test"), imgsz=640, verbose=False, stream=False, max_det=300)
sample = sorted((DATA_ROOT/"images"/"test").glob("*.jpg"))[:100]

for _ in range(5):
    model.predict(source=str(sample[0]), imgsz=640, verbose=False)   # warm up

start = time.perf_counter()
for path in sample:
    model.predict(source=str(path), imgsz=640, verbose=False)
elapsed = time.perf_counter() - start

print(f"{len(sample)/elapsed:.1f} fps on {torch.cuda.get_device_name(0)}")
print("PROXY ONLY (ADR-003). FR-D06's >=10 fps applies to the EDGE host, "
      "and a T4 is not it.")

## Export the weights

Download `best.pt` and `s11_detector_metrics.csv` from the output panel. Weights
go to Hugging Face, not into git (ADR-013 rev 2): LFS bandwidth is consumed by
every clone and CI checkout, and exhausting it reads to a teammate as a broken
clone rather than as a quota.

In [ ]:
best = Path("/kaggle/working/runs/s11_yolov8s_idd/weights/best.pt")
print("weights :", best, "|", round(best.stat().st_size/1e6, 1), "MB" if best.exists() else "MISSING")
print("metrics :", out)
print()
print("Next in the build log:")
print("  1. commit s11_detector_metrics.csv to experiments/results/")
print("  2. push best.pt to Hugging Face (ADR-013 rev 2)")
print("  3. RE-RUN THE P5 LABEL-NOISE PILOT with this detector.")
print("     The current figure (SNR 2.06-3.25, 11-26% of labels flippable) was")
print("     measured with stock COCO YOLOv8n, which cannot even name an")
print("     auto-rickshaw. Stock-vs-tuned is a publishable result and it")
print("     quantifies exactly what this training bought the corpus.")